# Multi-Subject NF fMRI — Yeo-17 Functional Connectivity Analysis
Parcel-level segregation, integration, normalized segregation, and participation coefficient across subjects, sessions, and runs.

## Imports

In [ ]:
import os
import glob
import pickle
import re
from collections import defaultdict
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def load_yeo_200(yeo=7):
    import os, pickle, pandas as pd, numpy as np
    Nregions    = 200
    pickle_file = "../../../Data/rsfMRI_utils.pkl"
    with open(pickle_file, "rb") as f:
        rsfMRI_utils = pickle.load(f)
    if yeo not in [7, 17]:
        raise NotImplementedError("Only Yeo 7 and 17 networks are implemented.")
    if yeo == 7:
        yeoOrder = rsfMRI_utils["yeo7"][Nregions]["yeoOrder"] - 1
        yeoROIs  = rsfMRI_utils["yeo7"][Nregions]["yeoROIs"]  - 1
        yeoROIs[-3:] = [7, 7, 7]
        yeoOrder = yeoOrder[:Nregions]
        yeoROIs  = yeoROIs[:Nregions]
        yeo_net  = {"VIS": 0, "SM": 1, "DA": 2, "VA": 3, "L": 4, "FP": 5, "DMN": 6}
        return yeoOrder, yeoROIs, yeo_net, rsfMRI_utils
    else:
        csv_path   = "../utils/matched_labels_exact.csv"
        df_mapping = pd.read_csv(csv_path)
        labels_from_csv    = [lbl.split("_", 1)[1] for lbl in df_mapping["label_df2"]]
        networks_from_csv  = np.array([r.split("_")[1] for r in labels_from_csv])
        yeo_order_17 = [
            "VisCent", "VisPeri", "SomMotA", "SomMotB",
            "DorsAttnA", "DorsAttnB", "SalVentAttnA", "SalVentAttnB",
            "LimbicB", "LimbicA", "ContA", "ContB", "ContC",
            "DefaultA", "DefaultB", "DefaultC", "TempPar",
        ]
        name_to_id = {name: idx for idx, name in enumerate(yeo_order_17)}
        yeoROIs    = np.array([name_to_id[n] for n in networks_from_csv], dtype=int)
        yeoOrder   = np.arange(len(yeoROIs))
        yeo_net    = name_to_id
        return yeoOrder, yeoROIs, yeo_net, rsfMRI_utils

## Configuration
Only this cell needs editing when you move to a different machine or dataset.

In [ ]:
# Root folder — one sub-directory per subject, e.g. data/NF/002GNV/
DATA_ROOT = "../../../Data/"

YEO17_NAMES = [
    "VisCent",
    "VisPeri",
    "SomMotA",
    "SomMotB",
    "DorsAttnA",
    "DorsAttnB",
    "SalVentAttnA",
    "SalVentAttnB",
    "LimbicA",
    "LimbicB",
    "ContA",
    "ContB",
    "ContC",
    "DefaultA",
    "DefaultB",
    "DefaultC",
    "TempPar",
]

YEO17_COLORS = [
    "#781286",  # 1  VisCent
    "#ff0101",  # 2  VisPeri
    "#4682b4",  # 3  SomMotA
    "#118ab2",  # 4  SomMotB
    "#2e8b57",  # 5  DorsAttnA
    "#00760e",  # 6  DorsAttnB
    "#dcf8a4",  # 7  SalVentAttnA
    "#c8e96e",  # 8  SalVentAttnB
    "#c9f8c9",  # 9  LimbicA
    "#e69422",  # 10 LimbicB
    "#f0e442",  # 11 ContA
    "#f0a800",  # 12 ContB
    "#f05a00",  # 13 ContC
    "#cd3d4f",  # 14 DefaultA
    "#e044af",  # 15 DefaultB
    "#ff6fc8",  # 16 DefaultC
    "#b15928",  # 17 TempPar
]

# Load Yeo-17 labels once (same for all subjects)
_, yeoROIs17, _, _ = load_yeo_200(yeo=17)
YEO17_LABELS = np.array(yeoROIs17).astype(int).ravel()  # (200,), labels 0-17

## 1. Data Loading
Auto-discover all subject folders, then load every session FC file for each subject.
The Yeo-17 atlas labels are loaded once from the shared parcellation (not per subject).

In [ ]:
def discover_subjects(data_root):
    """Return sorted list of subject IDs (= sub-folder names) in data_root."""
    return sorted(
        d for d in os.listdir(data_root)
        if os.path.isdir(os.path.join(data_root, d))
    )


def load_subject(data_root, subject_id):
    """
    Load one subject's session FC data.

    Returns
    -------
    sessions : dict  {session_id -> {run_key -> {"condition": str, "FC": ndarray}}}
    """
    subj_dir = os.path.join(data_root, subject_id)
    fc_files = sorted(glob.glob(os.path.join(subj_dir, f"{subject_id}_V*_FC_Schaefer.pkl")))
    print(f"  {subject_id}: {len(fc_files)} session file(s)")
    sessions = {}
    for fc_path in fc_files:
        session_id = os.path.basename(fc_path).split("_")[1]
        with open(fc_path, "rb") as f:
            data = pickle.load(f)
        run_map = {}
        for key, fc in data["FC_FD"].items():
            if not key.endswith("_200"):
                continue
            m = re.match(r"V\d+_Run(\d+)(?:_(Transfer|NoFeedback))?_200", key)
            if m is None:
                continue
            run_key   = f"Run{m.group(1)}"
            condition = m.group(2) if m.group(2) else "Feedback"
            run_map[run_key] = {"condition": condition, "FC": fc}
        sessions[session_id] = run_map
    return sessions


# --- Discover and load all subjects ---
subject_ids = discover_subjects(DATA_ROOT)
print(f"Subjects found: {subject_ids}\n")

# all_data[subject_id] = {"sessions": ..., "yeo_labels": ...}
all_data = {}
for sid in subject_ids:
    sessions = load_subject(DATA_ROOT, sid)
    all_data[sid] = {"sessions": sessions, "yeo_labels": YEO17_LABELS}

print(f"\nLoaded {len(all_data)} subject(s).")

In [ ]:
count = 0
subjects_with_run = []

for sid, subj_data in all_data.items():
    sessions = subj_data["sessions"]
    
    if "V03" in sessions and "Run8" in sessions["V03"]:
        count += 1
        subjects_with_run.append(sid)

print("Subjects with V03 Run8:", count)
print(subjects_with_run)

In [ ]:
for sid, subj_data in all_data.items():
    sessions = subj_data["sessions"]
    
    if "V03" in sessions and "Run8" in sessions["V03"]:
        del sessions["V03"]["Run8"]

In [ ]:
counts = defaultdict(int)

for sid, subj_data in all_data.items():
    for sess, runs in subj_data["sessions"].items():
        for run in runs.keys():
            counts[(sess, run)] += 1

rows = []
for (sess, run), n in counts.items():
    rows.append({"session": sess, "run": run, "n_subjects": n})

df_counts = pd.DataFrame(rows)
table = df_counts.pivot(index="session", columns="run", values="n_subjects")

print(table)

## 2. Metric Computation — Parcel-Level First
For each subject and session:
1. Fisher z-transform the raw parcel FC matrix.
2. Restrict to **cortical parcels (Yeo labels 1–17)** → ≈200×200 matrix.
3. For each network k:
   - **Segregation**: mean FC among all within-network parcel pairs (diagonal excluded).
   - **Integration**: mean FC between parcels in network k and parcels in all other networks.
   - **Normalized segregation**: `(seg − int) / (seg + int)`.
4. **Participation coefficient** (parcel level): proportion of each parcel's total connectivity that goes to other networks.

In [ ]:
def compute_yeo17_metrics(fc, yeo_labels):
    """
    Parcel-level Yeo-17 segregation / integration / participation coefficient.

    Parameters
    ----------
    fc         : (N, N) array  raw parcel FC (Pearson r)
    yeo_labels : (N,)   int array  Yeo-17 labels 0–17 in same parcel order

    Returns
    -------
    segregation            : (17,) array
    integration            : (17,) array
    normalized_segregation : (17,) array
    pc_parcel              : (~200,) array  participation coefficient per cortical parcel
    """
    # Step 1 – restrict to cortical parcels (Yeo 1-17)
    cortical_mask = (yeo_labels >= 1) & (yeo_labels <= 17)
    fc_cx  = np.array(fc[np.ix_(cortical_mask, cortical_mask)], dtype=float, copy=True)
    yeo_cx = yeo_labels[cortical_mask].astype(int)

    # Step 2 – Fisher z-transform, remove self-connections
    fc_cx = np.clip(fc_cx, -0.999999, 0.999999)
    fc_z  = np.arctanh(fc_cx)
    np.fill_diagonal(fc_z, np.nan)

    # Step 3 – per-network seg/int
    K = 17
    segregation = np.full(K, np.nan)
    integration = np.full(K, np.nan)

    for k in range(1, K + 1):
        mask_k     = (yeo_cx == k)
        mask_other = (yeo_cx != k)

        # within-network (segregation)
        if mask_k.sum() >= 2:
            within = fc_z[np.ix_(mask_k, mask_k)].copy()
            np.fill_diagonal(within, np.nan)
            segregation[k - 1] = np.nanmean(within)

        # between-network (integration)
        if mask_k.sum() > 0 and mask_other.sum() > 0:
            integration[k - 1] = np.nanmean(fc_z[np.ix_(mask_k, mask_other)])

    normalized_segregation = (segregation - integration) / (segregation + integration)

    # Step 4 – participation coefficient (parcel level)
    N = fc_z.shape[0]
    fc_work = np.nan_to_num(fc_z, nan=0.0)

    pc = np.full(N, np.nan)
    for i in range(N):
        k_i = np.sum(fc_work[i, :])
        if k_i == 0:
            continue
        sum_sq = sum((np.sum(fc_work[i, yeo_cx == s]) / k_i) ** 2 for s in range(1, 18))
        pc[i] = 1.0 - sum_sq

    return segregation, integration, normalized_segregation, pc


def compute_all_metrics(all_data):
    """Add segregation / integration / normalized_segregation / pc_parcel to every run entry in-place."""
    for subj in all_data.values():
        yeo = subj["yeo_labels"]
        for sess_runs in subj["sessions"].values():
            for rd in sess_runs.values():
                seg, intg, normseg, pc_parcel = compute_yeo17_metrics(rd["FC"], yeo)
                rd["segregation"]            = seg
                rd["integration"]            = intg
                rd["normalized_segregation"] = normseg
                rd["pc_parcel"]              = pc_parcel


compute_all_metrics(all_data)
print("Metrics computed for all subjects.")

## 3. Plotting Functions
Two reusable functions:
- `plot_runs_all_subjects` — 3×1 stacked axes, x-axis = every Session_Run across the dataset.
- `plot_session_averages_all_subjects` — same layout but x-axis = session (runs averaged within each session).

Both show **individual subjects** as lighter lines and the **group average** as a bold line.

In [ ]:
# -------------------------------------------------------------------------
# Helper: collect per-subject rows (list of dicts, one per session×run)
# -------------------------------------------------------------------------
def collect_rows(all_data, condition_filter=None):
    """
    Build a list of row-dicts from all subjects.
    Each row: subject, session, run, condition, segregation (17,), integration (17,),
              normalized_segregation (17,).

    condition_filter : str or None  – keep only this condition if given
    """
    all_rows = {}
    for sid, subj in all_data.items():
        rows = []
        for sess_id, sess_runs in subj["sessions"].items():
            for run_key in sorted(sess_runs.keys(), key=lambda x: int(x.replace("Run", ""))):
                rd = sess_runs[run_key]
                if condition_filter and rd["condition"] != condition_filter:
                    continue
                rows.append({
                    "subject":               sid,
                    "session":               sess_id,
                    "run":                   run_key,
                    "label":                 f"{sess_id}_{run_key}",
                    "condition":             rd["condition"],
                    "segregation":           rd["segregation"],
                    "integration":           rd["integration"],
                    "normalized_segregation": rd["normalized_segregation"],
                })
        all_rows[sid] = rows
    return all_rows


# -------------------------------------------------------------------------
# Plot 1: all runs, x = Session_Run
# -------------------------------------------------------------------------
def plot_runs_all_subjects(all_data, yeo_names, condition_filter=None, average_only=False, shade_conditions=False, show_subject_counts=False):
    """
    3×1 stacked plot (Segregation / Integration / Norm. Segregation).
    X-axis: every Session_Run label (union across subjects, in sorted order).
    Each subject is a semi-transparent line; group average is bold black.
    """
    subject_rows = collect_rows(all_data, condition_filter)

    # Build the union of all Session_Run labels in sorted order
    all_labels_set = set()
    for rows in subject_rows.values():
        for r in rows:
            all_labels_set.add(r["label"])
    all_labels = sorted(all_labels_set)
    label_to_x = {lbl: i for i, lbl in enumerate(all_labels)}
    x_all = np.arange(len(all_labels))

    # Map each label to its condition (take from any subject that has it)
    label_to_condition = {}
    for rows in subject_rows.values():
        for r in rows:
            label_to_condition[r["label"]] = r["condition"]

    shade_colors = {
        "Transfer":    ("orange", 0.18),
        "NoFeedback":  ("green",  0.18),
    }


    # Accumulate values per label for the group average
    # group_acc[label] = {metric: list of (17,) arrays}
    group_acc = {lbl: {"segregation": [], "integration": [], "normalized_segregation": []}
                 for lbl in all_labels}
    
    # Count number of subjects contributing to each label
    label_subjects = {lbl: set() for lbl in all_labels}

    for sid, rows in subject_rows.items():
        for r in rows:
            label_subjects[r["label"]].add(sid)

    label_counts = np.array([len(label_subjects[lbl]) for lbl in all_labels])
    
    for rows in subject_rows.values():
        for r in rows:
            lbl = r["label"]
            group_acc[lbl]["segregation"].append(r["segregation"])
            group_acc[lbl]["integration"].append(r["integration"])
            group_acc[lbl]["normalized_segregation"].append(r["normalized_segregation"])

    # Group average arrays  (n_labels × 17)
    seg_avg     = np.vstack([np.nanmean(group_acc[l]["segregation"],            axis=0) for l in all_labels])
    int_avg     = np.vstack([np.nanmean(group_acc[l]["integration"],            axis=0) for l in all_labels])
    normseg_avg = np.vstack([np.nanmean(group_acc[l]["normalized_segregation"], axis=0) for l in all_labels])

    cond_str = f" [{condition_filter}]" if condition_filter else ""
    fig, axes = plt.subplots(3, 1, figsize=(22, 16), sharex=True)

    if shade_conditions:
        for lbl in all_labels:
            cond = label_to_condition.get(lbl)
            if cond in shade_colors:
                color, alpha = shade_colors[cond]
                xi = label_to_x[lbl]
                for ax in axes:
                    ax.axvspan(xi - 0.5, xi + 0.5, color=color, alpha=alpha, zorder=0)
        # Add shading legend entries
        from matplotlib.patches import Patch
        shade_handles = [Patch(color=c, alpha=a, label=cond)
                         for cond, (c, a) in shade_colors.items()]
        axes[1].legend(handles=shade_handles, loc="center left", bbox_to_anchor=(1, 0.5))


    if not average_only:
        for sid, rows in subject_rows.items():
            if not rows:
                continue
            xs   = [label_to_x[r["label"]] for r in rows]
            segs = np.vstack([r["segregation"]            for r in rows])
            ints = np.vstack([r["integration"]            for r in rows])
            nrms = np.vstack([r["normalized_segregation"] for r in rows])

            for i in range(17):
                axes[0].plot(xs, segs[:, i], marker="o", alpha=0.35, linewidth=1)
                axes[1].plot(xs, ints[:, i], marker="o", alpha=0.35, linewidth=1)
                axes[2].plot(xs, nrms[:, i], marker="o", alpha=0.35, linewidth=1)

    # Group average — bold, labeled
    for i in range(17):
        axes[0].plot(x_all, seg_avg[:, i],     marker="o", linewidth=2, color=YEO17_COLORS[i], label=yeo_names[i])
        axes[1].plot(x_all, int_avg[:, i],     marker="o", linewidth=2, color=YEO17_COLORS[i], label=yeo_names[i])
        axes[2].plot(x_all, normseg_avg[:, i], marker="o", linewidth=2, color=YEO17_COLORS[i], label=yeo_names[i])

    axes[0].set_ylabel("Segregation")
    axes[0].set_title(f"Within-network connectivity (Segregation){cond_str}")

    axes[1].set_ylabel("Integration")
    axes[1].set_title(f"Between-network connectivity (Integration){cond_str}")

    axes[2].set_ylabel("Normalized segregation")
    axes[2].set_title(f"Normalized segregation across runs{cond_str}")
    axes[2].axhline(0, linestyle="--", linewidth=1)

    axes[2].set_xticks(x_all)
    axes[2].set_xticklabels(all_labels, rotation=90)
    axes[2].set_xlabel("Session / Run")

    if show_subject_counts:
        ax_count = axes[2].twinx()
        ax_count.plot(x_all, label_counts, color="black", linestyle="--", marker="s")
        ax_count.set_ylabel("Subjects")

    axes[0].legend(loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    plt.show()


# -------------------------------------------------------------------------
# Plot 2: session averages, x = session
# -------------------------------------------------------------------------
def plot_session_averages_all_subjects(all_data, yeo_names, condition_filter=None, average_only=False):
    """
    3×1 stacked plot of session-averaged metrics.
    X-axis: session labels (union across subjects).
    Each subject is a semi-transparent line; group average is bold black.
    """
    subject_rows = collect_rows(all_data, condition_filter)

    # Union of sessions
    all_sessions_set = set()
    for rows in subject_rows.values():
        for r in rows:
            all_sessions_set.add(r["session"])
    all_sessions = sorted(all_sessions_set)
    x_sessions = np.arange(len(all_sessions))

    # Per-subject session averages
    def session_avg_for_subject(rows):
        """Returns (n_sessions × 17) arrays for seg / int / normseg."""
        session_groups = defaultdict(list)
        for r in rows:
            session_groups[r["session"]].append(r)

        seg_sv, int_sv, normseg_sv = [], [], []
        for sess in all_sessions:
            sess_rows = session_groups.get(sess, [])
            if sess_rows:
                seg_sv.append(np.nanmean(np.vstack([r["segregation"]            for r in sess_rows]), axis=0))
                int_sv.append(np.nanmean(np.vstack([r["integration"]            for r in sess_rows]), axis=0))
                normseg_sv.append(np.nanmean(np.vstack([r["normalized_segregation"] for r in sess_rows]), axis=0))
            else:
                seg_sv.append(np.full(17, np.nan))
                int_sv.append(np.full(17, np.nan))
                normseg_sv.append(np.full(17, np.nan))

        return (np.vstack(seg_sv), np.vstack(int_sv), np.vstack(normseg_sv))

    per_subject = {sid: session_avg_for_subject(rows)
                   for sid, rows in subject_rows.items() if rows}

    # Group average across subjects
    seg_group     = np.nanmean(np.stack([v[0] for v in per_subject.values()]), axis=0)
    int_group     = np.nanmean(np.stack([v[1] for v in per_subject.values()]), axis=0)
    normseg_group = np.nanmean(np.stack([v[2] for v in per_subject.values()]), axis=0)

    cond_str = f" [{condition_filter}]" if condition_filter else ""
    fig, axes = plt.subplots(3, 1, figsize=(22, 16), sharex=True)

    # Individual subjects (faded)
    if not average_only:
        for sid, (seg_sv, int_sv, normseg_sv) in per_subject.items():
            for i in range(17):
                axes[0].plot(x_sessions, seg_sv[:, i],     marker="o", alpha=0.3, linewidth=1)
                axes[1].plot(x_sessions, int_sv[:, i],     marker="o", alpha=0.3, linewidth=1)
                axes[2].plot(x_sessions, normseg_sv[:, i], marker="o", alpha=0.3, linewidth=1)

    # Group average — bold, labeled
    for i in range(17):
        axes[0].plot(x_sessions, seg_group[:, i],     marker="o", linewidth=2, color=YEO17_COLORS[i], label=yeo_names[i])
        axes[1].plot(x_sessions, int_group[:, i],     marker="o", linewidth=2, color=YEO17_COLORS[i], label=yeo_names[i])
        axes[2].plot(x_sessions, normseg_group[:, i], marker="o", linewidth=2, color=YEO17_COLORS[i], label=yeo_names[i])

    axes[0].set_ylabel("Segregation")
    axes[0].set_title(f"Within-network connectivity (Segregation) — Session averages (all runs){cond_str}")

    axes[1].set_ylabel("Integration")
    axes[1].set_title(f"Between-network connectivity (Integration) — Session averages (all runs){cond_str}")

    axes[2].set_ylabel("Normalized segregation")
    axes[2].set_title(f"Normalized segregation across sessions — Session averages (all runs){cond_str}")
    axes[2].axhline(0, linestyle="--", linewidth=1)

    axes[2].set_xticks(x_sessions)
    axes[2].set_xticklabels(all_sessions)
    axes[2].set_xlabel("Session")

    axes[0].legend(loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    plt.show()


print("Plotting functions defined.")

## 4. All Runs — All Conditions
X-axis = every Session_Run in the dataset.  
Faded lines = individual subjects · Bold lines = group average.

In [ ]:
plot_runs_all_subjects(all_data, YEO17_NAMES)

### 4b. Group Average Only — All Runs, All Conditions, Transfer, No Feedback

In [ ]:
plot_runs_all_subjects(all_data, YEO17_NAMES, average_only=True, shade_conditions=True)

## 5. All Runs — Feedback Only

In [ ]:
plot_runs_all_subjects(all_data, YEO17_NAMES, condition_filter="Feedback")

### 5b. Group Average Only — All Runs, Feedback Only

In [ ]:
plot_runs_all_subjects(all_data, YEO17_NAMES, condition_filter="Feedback", average_only=True)

## 6. Session Averages — All Conditions
Runs are averaged within each session.  
Faded lines = individual subjects · Bold lines = group average.

In [ ]:
plot_session_averages_all_subjects(all_data, YEO17_NAMES)

### 6b. Group Average Only — Session Averages, All Conditions

In [ ]:
plot_session_averages_all_subjects(all_data, YEO17_NAMES, average_only=True)

## 7. Session Averages — Feedback Only

In [ ]:
plot_session_averages_all_subjects(all_data, YEO17_NAMES, condition_filter="Feedback")

### 7b. Group Average Only — Session Averages, Feedback Only

In [ ]:
plot_session_averages_all_subjects(all_data, YEO17_NAMES, condition_filter="Feedback", average_only=True)

## 8. Statistical testing

In [ ]:
# Use the group-average session values computed the same way as the plots
# Recompute them here cleanly for all conditions
subject_rows = collect_rows(all_data)
all_sessions_set = set()
for rows in subject_rows.values():
    for r in rows:
        all_sessions_set.add(r["session"])
all_sessions = sorted(all_sessions_set)

def session_avg_for_subject(rows, all_sessions):
    session_groups = defaultdict(list)
    for r in rows:
        session_groups[r["session"]].append(r)
    seg_sv, int_sv, normseg_sv = [], [], []
    for sess in all_sessions:
        sess_rows = session_groups.get(sess, [])
        if sess_rows:
            seg_sv.append(np.nanmean(np.vstack([r["segregation"] for r in sess_rows]), axis=0))
            int_sv.append(np.nanmean(np.vstack([r["integration"] for r in sess_rows]), axis=0))
            normseg_sv.append(np.nanmean(np.vstack([r["normalized_segregation"] for r in sess_rows]), axis=0))
        else:
            seg_sv.append(np.full(17, np.nan))
            int_sv.append(np.full(17, np.nan))
            normseg_sv.append(np.full(17, np.nan))
    return np.vstack(seg_sv), np.vstack(int_sv), np.vstack(normseg_sv)

per_subject = {sid: session_avg_for_subject(rows, all_sessions)
               for sid, rows in subject_rows.items() if rows}

seg_group     = np.nanmean(np.stack([v[0] for v in per_subject.values()]), axis=0)  # n_sessions x 17
int_group     = np.nanmean(np.stack([v[1] for v in per_subject.values()]), axis=0)
normseg_group = np.nanmean(np.stack([v[2] for v in per_subject.values()]), axis=0)

x = np.arange(len(all_sessions))  # session index as the "time" variable

print(f"{'Network':<20} {'Seg ρ':>8} {'Seg p':>10} {'Int ρ':>8} {'Int p':>10} {'NormSeg ρ':>10} {'NormSeg p':>10}")
print("-" * 80)
for i, net in enumerate(YEO17_NAMES):
    rho_seg,    p_seg    = spearmanr(x, seg_group[:, i])
    rho_int,    p_int    = spearmanr(x, int_group[:, i])
    rho_nrm,    p_nrm    = spearmanr(x, normseg_group[:, i])
    def sig_stars(p):
        if p < 0.001: return "***"
        if p < 0.01:  return "**"
        if p < 0.05:  return "*"
        return ""
    def fmt_p(p):
        stars = sig_stars(p)
        return f"{p:6.3f}{stars:<3}"  # stars left-aligned in a 3-char slot


    print(f"{net:<20} {rho_seg:>+8.3f} {fmt_p(p_seg)}  {rho_int:>+8.3f} {fmt_p(p_int)}  {rho_nrm:>+8.3f} {fmt_p(p_nrm)}")


In [ ]:
# Collect all 51 p-values in one list (17 networks × 3 metrics)
results = []
for i, net in enumerate(YEO17_NAMES):
    rho_seg, p_seg = spearmanr(x, seg_group[:, i])
    rho_int, p_int = spearmanr(x, int_group[:, i])
    rho_nrm, p_nrm = spearmanr(x, normseg_group[:, i])
    results.append((net, rho_seg, p_seg, rho_int, p_int, rho_nrm, p_nrm))

all_p = [r[2] for r in results] + [r[4] for r in results] + [r[6] for r in results]

# FDR correction (Benjamini-Hochberg)
_, p_adj, _, _ = multipletests(all_p, method="fdr_bh")

p_adj_seg     = p_adj[0:17]
p_adj_int     = p_adj[17:34]
p_adj_normseg = p_adj[34:51]

print(f"{'Network':<20} {'Seg ρ':>8} {'Seg p_adj':<12}  {'Int ρ':>8} {'Int p_adj':<12}  {'NormSeg ρ':>8} {'NormSeg p_adj':<12}")
print("-" * 95)
for i, (net, rho_seg, _, rho_int, _, rho_nrm, _) in enumerate(results):
    print(f"{net:<20} {rho_seg:>+8.3f} {fmt_p(p_adj_seg[i]):<12}  "
          f"{rho_int:>+8.3f} {fmt_p(p_adj_int[i]):<12}  "
          f"{rho_nrm:>+8.3f} {fmt_p(p_adj_normseg[i]):<12}")

print("\nFDR-corrected (Benjamini-Hochberg)   * p < 0.05   ** p < 0.01   *** p < 0.001")


## 10. Per-Subject Plots
Run the same two plot types for a single subject at a time.  
Change `SUBJECT_ID` to inspect any subject.

In [ ]:
SUBJECT_ID = subject_ids[0]   # change to any subject ID, e.g. "002GNV"

single_subj_data = {SUBJECT_ID: all_data[SUBJECT_ID]}

print(f"--- {SUBJECT_ID} | All runs ---")
plot_runs_all_subjects(single_subj_data, YEO17_NAMES, shade_conditions=True)

print(f"--- {SUBJECT_ID} | Session averages ---")
plot_session_averages_all_subjects(single_subj_data, YEO17_NAMES)

## 11. Export
Save every run-level metric to a CSV for downstream analysis.

In [ ]:
import pandas as pd

export_rows = []
for sid, subj in all_data.items():
    for sess_id, sess_runs in subj["sessions"].items():
        for run_key in sorted(sess_runs.keys(), key=lambda x: int(x.replace("Run", ""))):
            rd = sess_runs[run_key]
            for net_idx, net_name in enumerate(YEO17_NAMES):
                export_rows.append({
                    "subject":               sid,
                    "session":               sess_id,
                    "run":                   run_key,
                    "condition":             rd["condition"],
                    "network":               net_name,
                    "segregation":           rd["segregation"][net_idx],
                    "integration":           rd["integration"][net_idx],
                    "normalized_segregation": rd["normalized_segregation"][net_idx],
                })

df_export = pd.DataFrame(export_rows)
out_path = os.path.join(DATA_ROOT, "all_subjects_yeo17_metrics.csv")
df_export.to_csv(out_path, index=False)
print(f"Saved:")
df_export.head(17)

## 12. Data structure inspection

In [ ]:
# Top level
print("Subjects:", list(all_data.keys()))
print()

# Drill into first subject
sid = list(all_data.keys())[0]
subj = all_data[sid]
print(f"Subject '{sid}' keys:", list(subj.keys()))
print(f"  yeo_labels shape: {subj['yeo_labels'].shape}")
print(f"  yeo_labels unique values: {np.unique(subj['yeo_labels'])}")
print()

# Sessions
print(f"  Sessions: {list(subj['sessions'].keys())}")
print()

# Drill into first session
sess_id = list(subj['sessions'].keys())[0]
sess = subj['sessions'][sess_id]
print(f"  Session '{sess_id}' runs: {list(sess.keys())}")
print()

# Drill into first run
run_id = list(sess.keys())[0]
run = sess[run_id]
print(f"  Run '{run_id}' keys: {list(run.keys())}")
for k, v in run.items():
    shape = getattr(v, 'shape', None)
    if shape:
        print(f"    {k}: {type(v).__name__}  shape={shape}  min={np.nanmin(v):.3f}  max={np.nanmax(v):.3f}")
    else:
        print(f"    {k}: {v}")